# تست کوچک: بررسی Point Group Symmetry Operations روی چند ماده

## هدف
قبل از طراحی augmentation فیزیکی کامل (که برای هر تقارن نقطه‌ای کریستال، یک نمونه‌ی
transform‌شده‌ی جدید می‌سازد)، باید بفهمیم:
1. هر ماده‌ی MAX phase معمولاً چند عملیات تقارن نقطه‌ای دارد؟ (برای تخمین این‌که augmentation
   چه‌قدر داده‌ی مؤثر اضافه می‌کند)
2. آیا phonopy/spglib مستقیماً این عملیات‌ها را به‌صورت ماتریس‌های قابل‌استفاده برمی‌گرداند؟
3. آیا این تقارن‌ها بین مواد مختلف یکسان هستند یا متفاوت؟ (مواد MAX phase همگی ساختار
   هگزاگونال P6₃/mmc دارند، پس انتظار تقارن مشابه بین همه‌ی مواد را داریم -- که اگر درست
   باشد، باز هم augmentation معنادار است چون نمونه‌های واقعی و متفاوت تولید می‌کند، حتی
   اگر خود عملیات‌های تقارن بین مواد یکسان باشند)

این یک تست سبک، بدون آموزش مدل، فقط برای بررسی ساختار symmetry است.

In [ ]:
!pip install -q phonopy -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("Installed")

In [ ]:
import numpy as np
import yaml
from pathlib import Path
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS

FC_DIR     = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
BANDS_DIR  = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'

FC_DIR_PATH = Path(FC_DIR)
BANDS_DIR_PATH = Path(BANDS_DIR)

fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
band_yaml_files = {f.stem: f for f in BANDS_DIR_PATH.glob('*.yaml')}

common = sorted(set(fc_files) & set(band_yaml_files))
print(f"Number of materials found: {len(common)}")

test_formulas = common[:5] + common[len(common)//2:len(common)//2+3] + common[-3:]
test_formulas = list(dict.fromkeys(test_formulas))
print(f"Test materials ({len(test_formulas)}): {test_formulas}")

## ساخت PhonopyAtoms برای هر ماده (بدون نیاز به کشف supercell، فقط unit cell لازم است)

In [ ]:
def load_unitcell(yaml_path):
    with open(yaml_path) as f:
        data = yaml.safe_load(f)
    lattice = np.array(data['lattice'])
    symbols = [p['symbol'] for p in data['points']]
    frac_coords = np.array([p['coordinates'] for p in data['points']])
    unitcell = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)
    return unitcell

print("load_unitcell ready")

## بررسی Point Group Symmetry با spglib (از طریق phonopy)

phonopy داخلش از spglib استفاده می‌کند که می‌تواند مستقیماً عملیات‌های تقارن (چرخش +
انتقال) را به‌صورت ماتریس برگرداند.

In [ ]:
import spglib

for formula in test_formulas:
    print(f"\n{'='*60}")
    print(f"Material: {formula}")
    try:
        unitcell = load_unitcell(band_yaml_files[formula])

        cell_tuple = (unitcell.cell, unitcell.scaled_positions, unitcell.numbers)
        dataset = spglib.get_symmetry_dataset(cell_tuple, symprec=1e-3)

        if dataset is None:
            print("  spglib could not determine symmetry")
            continue

        n_ops = len(dataset.rotations)
        print(f"  Space group: {dataset.international} (number {dataset.number})")
        print(f"  Point group: {dataset.pointgroup}")
        print(f"  Number of symmetry operations: {n_ops}")

        # show first few rotation matrices (these act on fractional coordinates)
        print(f"  First 3 rotation matrices:")
        for i in range(min(3, n_ops)):
            print(f"    op {i}: rotation=\n{dataset.rotations[i]}\n      translation={dataset.translations[i]}")

    except Exception as e:
        print(f"  Unexpected error: {e}")

## نتیجه‌گیری این تست

به موارد زیر در خروجی بالا دقت کنید:
1. **چند عملیات تقارن** هر ماده دارد؟ (هرچه بیشتر، augmentation مؤثرتر -- چون می‌توانیم
   به همان نسبت نمونه‌ی جدید معتبر بسازیم)
2. **آیا گروه فضایی بین مواد یکسان است؟** (احتمالاً بله، چون همه از خانواده‌ی MAX phase
   با ساختار هگزاگونال هستند)
3. **ماتریس‌های rotation چه شکلی دارند؟** (این‌ها ماتریس‌های ۳×۳ صحیح هستند که روی
   fractional coordinates عمل می‌کنند -- این دقیقاً همان چیزی است که برای augmentation به
   آن نیاز داریم: لاتیس، موقعیت اتم‌ها، و IFC را با این ماتریس‌ها transform می‌کنیم)

**لطفاً کل خروجی این تست را برای من بفرستید.**